# 🌿 Satellite Vegetation Health Classification with YOLO26

**Training-course final project — Google Colab**

### Project idea
Classify a Sentinel-2 satellite image patch as:

- **healthy** vegetation
- **stressed** vegetation

We use **Ultralytics YOLO26n Classification** because it is small and simple enough for Colab.

> **Important scientific limitation:** EuroSAT does **not** contain true “healthy/stressed” labels.  
> In this beginner project, we create **proxy labels from NDVI** using Sentinel-2 Red and NIR bands. Higher NDVI is treated as healthier vegetation and lower NDVI as more stressed vegetation. This is a demonstration project, not a field-validated crop-diagnosis system.

### Simple workflow

`EuroSAT multispectral → keep vegetation classes → calculate NDVI → create healthy/stressed folders → train YOLO26n-cls → evaluate → predict → export ONNX`


## 1. Setup

In Colab, select **Runtime → Change runtime type → T4 GPU** if available.


In [ ]:
!pip -q install -U ultralytics datasets pillow scikit-learn

from ultralytics import YOLO
from datasets import load_dataset
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import random, shutil, os

print("Setup complete.")


## 2. Download the satellite dataset

We use **EuroSAT**, based on Sentinel-2 satellite images.  
The original EuroSAT project is available on GitHub: `phelber/EuroSAT`.

For an easy Colab download, this notebook uses a public Hugging Face mirror of the **13-band multispectral** data.

We only keep vegetation-related land-cover classes:
- AnnualCrop
- Forest
- HerbaceousVegetation
- Pasture
- PermanentCrop


In [ ]:
# Public mirror of EuroSAT multispectral Sentinel-2 data
ds = load_dataset("giswqs/EuroSAT_MS", split="train")

print(ds)
print("Columns:", ds.column_names)


## 3. Create healthy/stressed proxy labels using NDVI

NDVI is:

**NDVI = (NIR - Red) / (NIR + Red)**

For Sentinel-2:
- Red = B04
- NIR = B08

To keep the course project simple, we calculate the **mean NDVI** for each vegetation image.  
Then we use the **median NDVI of our selected samples** as the split:

- NDVI ≥ median → **healthy**
- NDVI < median → **stressed**

This creates balanced classes, but remember these are **proxy labels**, not expert ground-truth stress labels.


In [ ]:
# Keep the notebook fast for a training course.
# Increase MAX_IMAGES later if you want a larger experiment.
MAX_IMAGES = 4000

vegetation_classes = {
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Pasture", "PermanentCrop"
}

samples = []

for item in ds:
    # The mirror provides the original filename, e.g. AnnualCrop/...
    class_name = item["filename"].split("/")[0]
    if class_name not in vegetation_classes:
        continue

    img = np.asarray(item["image"], dtype=np.float32)  # expected shape: (13, 64, 64)

    red = img[3]   # B04
    nir = img[7]   # B08
    ndvi = (nir - red) / (nir + red + 1e-6)
    mean_ndvi = float(np.nanmean(ndvi))

    samples.append((img, mean_ndvi))

    if len(samples) >= MAX_IMAGES:
        break

ndvi_values = np.array([x[1] for x in samples])
threshold = float(np.median(ndvi_values))

print("Vegetation images:", len(samples))
print("NDVI median threshold:", round(threshold, 3))


## 4. Convert Sentinel-2 patches to RGB and make YOLO folders

YOLO image classification expects folders like:

```text
vegetation_dataset/
  train/
    healthy/
    stressed/
  val/
    healthy/
    stressed/
  test/
    healthy/
    stressed/
```

We convert Sentinel-2 B04/B03/B02 to an RGB image and split the data 70% / 15% / 15%.


In [ ]:
root = Path("/content/vegetation_dataset")
if root.exists():
    shutil.rmtree(root)

for split in ["train", "val", "test"]:
    for label in ["healthy", "stressed"]:
        (root / split / label).mkdir(parents=True, exist_ok=True)

def to_rgb(img13):
    # Sentinel-2: B04=red, B03=green, B02=blue
    rgb = np.stack([img13[3], img13[2], img13[1]], axis=-1)
    # Simple percentile stretch for visualization/training
    lo, hi = np.percentile(rgb, (2, 98))
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)
    return (rgb * 255).astype(np.uint8)

random.seed(42)
indices = list(range(len(samples)))
random.shuffle(indices)

n = len(indices)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

split_indices = {
    "train": indices[:train_end],
    "val": indices[train_end:val_end],
    "test": indices[val_end:]
}

for split, ids in split_indices.items():
    for j, idx in enumerate(ids):
        img13, mean_ndvi = samples[idx]
        label = "healthy" if mean_ndvi >= threshold else "stressed"
        rgb = to_rgb(img13)
        Image.fromarray(rgb).save(root / split / label / f"{split}_{j:05d}.jpg")

for split in ["train", "val", "test"]:
    print(split,
          "healthy =", len(list((root/split/"healthy").glob("*.jpg"))),
          "stressed =", len(list((root/split/"stressed").glob("*.jpg"))))


## 5. View example satellite images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, label in enumerate(["healthy", "stressed"]):
    files = list((root/"train"/label).glob("*.jpg"))
    for col, f in enumerate(random.sample(files, min(4, len(files)))):
        axes[row, col].imshow(Image.open(f))
        axes[row, col].set_title(label)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


## 6. Train the latest released Ultralytics YOLO model

We use **YOLO26n-cls**:
- `26` = current released YOLO generation
- `n` = nano, the smallest model
- `cls` = image classification

For a classroom demo, start with **10 epochs**. Increase to 20–50 for a stronger experiment.


In [ ]:
model = YOLO("yolo26n-cls.pt")

train_results = model.train(
    data=str(root),
    epochs=10,
    imgsz=224,
    batch=32,
    device=0,       # GPU in Colab
    project="/content/runs",
    name="vegetation_stress"
)


## 7. Evaluate the model

In [ ]:
best_model = YOLO("/content/runs/vegetation_stress/weights/best.pt")

metrics = best_model.val(data=str(root), split="test")
print(metrics)


### Evaluation metrics to report

For this binary classification project, report:
- **Top-1 accuracy**
- Confusion matrix
- Example successful prediction
- Example failure case

Ultralytics saves training/validation plots in the run folder.


In [ ]:
# Show result images generated by Ultralytics, if available
run_dir = Path("/content/runs/vegetation_stress")

for name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png"]:
    p = run_dir / name
    if p.exists():
        display(Image.open(p))


## 8. Make predictions and find a success + failure case

In [ ]:
test_files = list((root/"test"/"healthy").glob("*.jpg")) + list((root/"test"/"stressed").glob("*.jpg"))
random.shuffle(test_files)

success = None
failure = None

for f in test_files:
    true_label = f.parent.name
    result = best_model.predict(str(f), verbose=False)[0]
    pred_id = int(result.probs.top1)
    pred_label = result.names[pred_id]
    conf = float(result.probs.top1conf)

    item = (f, true_label, pred_label, conf)

    if pred_label == true_label and success is None:
        success = item
    if pred_label != true_label and failure is None:
        failure = item
    if success and failure:
        break

def show_case(item, title):
    if item is None:
        print(title, ": not found in checked test images.")
        return
    f, true_label, pred_label, conf = item
    plt.figure(figsize=(4,4))
    plt.imshow(Image.open(f))
    plt.axis("off")
    plt.title(f"{title}\nTrue: {true_label} | Pred: {pred_label} ({conf:.2f})")
    plt.show()

show_case(success, "Successful prediction")
show_case(failure, "Failure case")


## 9. Predict a new image

Upload any RGB satellite crop/vegetation image.  
The model will return **healthy** or **stressed** according to the proxy labels learned in this project.


In [ ]:
from google.colab import files

uploaded = files.upload()
new_image = next(iter(uploaded.keys()))

prediction = best_model.predict(new_image, imgsz=224)
prediction[0].show()


## 10. Export to ONNX

This satisfies the optional deployment/optimization part of the final-project requirements.


In [ ]:
best_model.export(format="onnx", imgsz=224)


## 11. Real-application idea

A simple real system could:

1. Receive a new Sentinel-2 image patch.
2. Convert the required bands to RGB.
3. Run the trained YOLO26 classifier.
4. Display **healthy** or **stressed** with confidence.
5. Flag stressed regions for inspection by an agronomist.

### Failure cases / limitations
- NDVI is only a **proxy** for vegetation condition.
- Low NDVI can be caused by sparse vegetation, soil, crop stage, clouds, shadows, or harvest — not only stress.
- EuroSAT was created for **land-cover classification**, not crop-stress diagnosis.
- A stronger final system should use field-validated stress labels, multiple dates, and multispectral bands directly.

### Future improvements
- Use true ground-truth crop-stress labels.
- Add temporal Sentinel-2 observations.
- Use NDVI, NDMI, red-edge and SWIR bands.
- Compare YOLO26 with a multispectral CNN.
- Train on more images and more epochs.
